# BBBP general BFS single setting — Q=16, T=23

This notebook is the local executable code for this minimum experiment folder.
It follows the original `GPU-simulate/BACE` per-setting style, but uses the
strict FINAL protocol: task-independent local-BFS tokens without global generic
priors, QRC/CRC dimensional alignment, and a single unregularized linear readout selected by validation
`AUC > loss > ACC`.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
import torch

SETTING_DIR = Path(r"/home/ubuntu/Workspace/Fei/wlf/Article/QRC/FINAL/2_linear_readout/general/BBBP/02_fixed_Q_T_scan/Q16/T23").resolve()
PLAN_PATH = SETTING_DIR / "setting_plan.json"
PROJECT_ROOT = Path(r"/home/ubuntu/Workspace/Fei/wlf/Article/QRC/FINAL/2_linear_readout").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from core.linear_readout import run_setting_plan, plot_single_setting_summary

print("setting:", SETTING_DIR)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())


In [ ]:
RUN_QRC = True   # set False only for CPU/debugging of CRC/raw
FORCE = False   # resume old QRC/controls and backfill only missing methods

plan = json.loads(PLAN_PATH.read_text(encoding="utf-8"))
display(pd.Series(plan, name="setting_plan"))


## Protocol locked in this folder

- Token stream: general, task-independent local-BFS event stream with no generic molecule-level descriptors, canonical ranks, full-graph ring sizes or neighbor aggregate statistics.
- Token slice: first `T` molecular events and first `3Q` token coordinates.
- QRC: `W=0.4`, `g=0.84`, `L=3`, tanh injection preprocessing, Z + all-ZZ readout.
- CRC: old scalar-Q reservoir, same input preprocessing, same number of reservoir layers, same output dimension as QRC.
- Raw-full baseline: direct flattening of the original `T × 3Q` token tensor.
- CRC3-z/zz matched: Q-by-3 classical state with direct z and all zz products; exactly Q(Q+1)/2 outputs and no PCA.
- Standard ESN: fixed Gaussian echo-state network with spectral radius 0.9 and direct QRC-dimension state; no PCA.
- Raw-PCA baseline: PCA fitted on each train split only, then compressed/padded to the QRC/CRC readout dimension.
- Classifier: exactly one `torch.nn.Linear(d, 1)`, no standardization, no hidden layer, no regularization.
- Seeds/splits: `0,10,...,90`; QRC/CRC parameters, scaffold split and training all change with seed.


In [ ]:
metrics, summary = run_setting_plan(
    PLAN_PATH,
    run_qrc=RUN_QRC,
    force=FORCE,
)

display(summary)
display(metrics.sort_values(["method", "seed"]).tail(12))


In [ ]:
fig = plot_single_setting_summary(SETTING_DIR, plan["dataset"])
plt.show()

summary_path = SETTING_DIR / "performance_summary.csv"
pred_path = SETTING_DIR / "test_predictions.csv"
print("summary:", summary_path)
print("predictions:", pred_path)
